#Preparation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [ ]:
# Load data
data = pd.read_csv(
    "water-treatment.data",
    header=None
)

print(data.shape)
data.head()

(527, 39)


,0,1,2,3,4,5,6,7,8,9,...,29,30,31,32,33,34,35,36,37,38
0,D-1/3/90,44101,1.50,7.8,?,407,166,66.3,4.5,2110,...,2000,?,58.8,95.5,?,70.0,?,79.4,87.3,99.6
1,D-2/3/90,39024,3.00,7.7,?,443,214,69.2,6.5,2660,...,2590,?,60.7,94.8,?,80.8,?,79.5,92.1,100
2,D-4/3/90,32229,5.00,7.6,?,528,186,69.9,3.4,1666,...,1888,?,58.2,95.6,?,52.9,?,75.8,88.7,98.5
3,D-5/3/90,35023,3.50,7.9,205,588,192,65.6,4.5,2430,...,1840,33.1,64.2,95.3,87.3,72.3,90.2,82.3,89.6,100
4,D-6/3/90,36924,1.50,8.0,242,496,176,64.8,4.0,2110,...,2120,?,62.7,95.6,?,71.0,92.1,78.2,87.5,99.5


In [ ]:
X_raw = data.copy()

n_days, n_vars = X_raw.shape
print("Days:", n_days)
print("Variables:", n_vars)

Days: 527
Variables: 39


In [ ]:
W = 14  # window size

In [ ]:
# Convert everything to numeric, force errors to NaN
X_raw = data.apply(pd.to_numeric, errors="coerce")

# Check how many NaNs we have
# Drop raw columns that are entirely NaN
raw_all_nan = X_raw.columns[X_raw.isna().all()]
print("Raw columns that are all NaN:", list(raw_all_nan))

X_raw = X_raw.drop(columns=raw_all_nan)

# Impute remaining
X_raw = X_raw.fillna(X_raw.mean())

Raw columns that are all NaN: [0]


In [ ]:
features = []

for t in range(W-1, len(X_raw)):
    row_features = []

    window = X_raw.iloc[t-W+1 : t+1]

    for col in X_raw.columns:
        x = window[col].values

        last = x[-1]
        mean = x.mean()
        std = x.std()
        diff = x[-1] - x[-2]

        time = np.arange(W)
        slope = np.polyfit(time, x, 1)[0]

        row_features.extend([last, mean, std, slope, diff])

    features.append(row_features)

X_feat = np.array(features)
print(X_feat.shape)

(514, 190)


In [ ]:
feature_names = []

for col in X_raw.columns:
    feature_names.extend([
        f"var{col}_last",
        f"var{col}_mean14",
        f"var{col}_std14",
        f"var{col}_trend14",
        f"var{col}_diff1"
    ])

X_feat = pd.DataFrame(X_feat, columns=feature_names)
X_feat.head()

,var1_last,var1_mean14,var1_std14,var1_trend14,var1_diff1,var2_last,var2_mean14,var2_std14,var2_trend14,var2_diff1,...,var37_last,var37_mean14,var37_std14,var37_trend14,var37_diff1,var38_last,var38_mean14,var38_std14,var38_trend14,var38_diff1
0,40376.0,38573.857143,4166.362405,263.578022,-2535.0,2.359065,2.732790,1.586577,-0.143367,1.659065,...,43.7,71.928571,28.440701,-5.294945,33.4,100.0,90.735714,20.923555,-2.178681,4.6
1,40923.0,38346.857143,3939.432680,512.909890,547.0,3.500000,2.875647,1.558944,-0.162089,1.140935,...,78.2,71.278571,28.184803,-4.609011,34.5,99.2,90.707143,20.911701,-1.644615,-0.8
2,43830.0,38690.142857,4185.214790,691.894505,2907.0,1.500000,2.768505,1.597778,-0.197293,-2.000000,...,90.7,71.178571,28.113143,-3.367692,12.5,100.0,90.707143,20.911701,-1.072747,0.8
3,39165.0,39185.571429,3782.168213,492.457143,-4665.0,1.200000,2.497076,1.516334,-0.168542,-0.300000,...,92.1,71.421429,28.277658,-2.192308,1.4,100.0,90.814286,20.955151,-0.550330,0.0
4,35791.0,39240.428571,3726.556563,258.241758,-3374.0,1.200000,2.332790,1.523352,-0.172538,0.000000,...,89.4,71.407143,28.268520,-1.079341,-2.7,99.4,90.771429,20.936927,-0.002198,-0.6


In [ ]:
print("Final feature matrix shape:", X_feat.shape)

Final feature matrix shape: (514, 190)


#Logistic Regression

In [ ]:
# TEMPORAL: just to test

np.random.seed(0)
y = np.zeros(len(X_feat))
failure_days = np.random.choice(len(y), size=50, replace=False)
y[failure_days] = 1

print("Positive samples:", int(y.sum()))

Positive samples: 50


In [ ]:
# Temporal split (60 / 20 / 20)
n = len(X_feat)

train_end = int(0.6 * n)
val_end   = int(0.8 * n)

X_train = X_feat.iloc[:train_end]
X_val   = X_feat.iloc[train_end:val_end]
X_test  = X_feat.iloc[val_end:]

y_train = y[:train_end]
y_val   = y[train_end:val_end]
y_test  = y[val_end:]

print(X_train.shape, X_val.shape, X_test.shape)

(308, 190) (103, 190) (103, 190)


In [ ]:
# Normalization (training only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=2000, n_jobs=-1)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

y_prob = model.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_prob > 0.5).astype(int)

roc  = roc_auc_score(y_test, y_prob)
pr   = average_precision_score(y_test, y_prob)
f1   = f1_score(y_test, y_pred)

print("ROC-AUC:", roc)
print("PR-AUC :", pr)
print("F1     :", f1)

ROC-AUC: 0.1563573883161512
PR-AUC : 0.03872344850264596
F1     : 0.02631578947368421


# Constructing real labels

In [ ]:
with open("water-treatment.names", "r") as f:
    names_text = f.read()

print(names_text[:1000])  # just to check

1. Title: Faults in a urban waste water treatment plant

2. Source Information:
   -- Creators: Manel Poch (igte2@cc.uab.es)
         Unitat d'Enginyeria Quimica
         Universitat Autonoma de Barcelona. Bellaterra. Barcelona; Spain
   -- Donor: Javier Bejar and Ulises Cortes (bejar@lsi.upc.es)
         Dept. Llenguatges i Sistemes Informatics;
         Universitat Politecnica de Catalunya. Barcelona; Spain
   -- Date: June, 1993

3. Past Usage:
   1. J. De Gracia.
      ``Avaluacio de tecniques de classificacio per a la gestio de 
        Bioprocessos: Aplicacio a un reactor de fangs activats''
         Master Thesis. Dept. de Quimica. Unitat d'Enginyeria Quimica.
       Universitat Autonoma de Barcelona. Bellaterra (Barcelona). 1993.
         -- Results:
              Comparison between the classification of plant situations using 
             cluster analysis and conceptual clustering. The induced classes
             are exposed and contrasted. 


   2. J. Bejar, U. Cort\'es and

In [ ]:
# Compute day-to-day change magnitude
daily_diff = np.linalg.norm(X_raw.diff().fillna(0).values, axis=1)

# Define failures as extreme changes (top 10%)
threshold = np.percentile(daily_diff, 90)

failure_day = (daily_diff > threshold).astype(int)

print("Number of failure days:", failure_day.sum())

Number of failure days: 53


In [ ]:
failure_day = failure_day[W-1:]
print(len(failure_day), X_feat.shape[0])

514 514


In [ ]:
def make_early_warning_labels(failure_series, H):
    y = np.zeros(len(failure_series))
    for t in range(len(failure_series)):
        if t + H < len(failure_series):
            y[t] = failure_series[t+1 : t+H+1].max()
    return y

y_H1 = make_early_warning_labels(failure_day, H=1)
y_H3 = make_early_warning_labels(failure_day, H=3)
y_H7 = make_early_warning_labels(failure_day, H=7)

print("H=1 positives:", y_H1.sum())
print("H=3 positives:", y_H3.sum())
print("H=7 positives:", y_H7.sum())

H=1 positives: 52.0
H=3 positives: 127.0
H=7 positives: 230.0


## y = y_H1

In [ ]:
y = y_H1  # real labels

# Temporal split
n = len(X_feat)
train_end = int(0.6 * n)
val_end   = int(0.8 * n)

X_train = X_feat.iloc[:train_end]
X_val   = X_feat.iloc[train_end:val_end]
X_test  = X_feat.iloc[val_end:]

y_train = y[:train_end]
y_val   = y[train_end:val_end]
y_test  = y[val_end:]

# Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# Train
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    n_jobs=-1
)
model.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=2000, n_jobs=-1)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

y_prob = model.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_prob > 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC :", average_precision_score(y_test, y_prob))
print("F1     :", f1_score(y_test, y_pred))

ROC-AUC: 0.6166666666666667
PR-AUC : 0.04864824717765895
F1     : 0.04


## y = y_H3

In [ ]:
y = y_H3  # real labels

# Temporal split
n = len(X_feat)
train_end = int(0.6 * n)
val_end   = int(0.8 * n)

X_train = X_feat.iloc[:train_end]
X_val   = X_feat.iloc[train_end:val_end]
X_test  = X_feat.iloc[val_end:]

y_train = y[:train_end]
y_val   = y[train_end:val_end]
y_test  = y[val_end:]

# Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# Train
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    n_jobs=-1
)
model.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=2000, n_jobs=-1)

In [ ]:
y_prob = model.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_prob > 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC :", average_precision_score(y_test, y_prob))
print("F1     :", f1_score(y_test, y_pred))

ROC-AUC: 0.7
PR-AUC : 0.10709133684743441
F1     : 0.1568627450980392


##y = y_H7

In [ ]:
y = y_H7  # real labels

# Temporal split
n = len(X_feat)
train_end = int(0.6 * n)
val_end   = int(0.8 * n)

X_train = X_feat.iloc[:train_end]
X_val   = X_feat.iloc[train_end:val_end]
X_test  = X_feat.iloc[val_end:]

y_train = y[:train_end]
y_val   = y[train_end:val_end]
y_test  = y[val_end:]

# Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# Train
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    n_jobs=-1
)
model.fit(X_train_scaled, y_train)

LogisticRegression(class_weight='balanced', max_iter=2000, n_jobs=-1)

In [ ]:
y_prob = model.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_prob > 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("PR-AUC :", average_precision_score(y_test, y_prob))
print("F1     :", f1_score(y_test, y_pred))

ROC-AUC: 0.6016548463356974
PR-AUC : 0.1102047859798684
F1     : 0.20512820512820512
